# 02 — Dữ liệu thị trường, vĩ mô và tài chính

Dự báo log return phiên kế tiếp, sau đó suy ra giá và hướng. Chạy các ô từ trên xuống. Dữ liệu dừng tại 30/09/2025.

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Ghép từ ngày có thể biết

Nguồn vĩ mô là các bản xuất đã lưu; `--from-cache` không phải tải website mới. Báo cáo quý trễ 45 ngày, GDP trễ 180 ngày; cổ phiếu lưu hành theo mốc công bố riêng + 1 ngày. Không điền ngược. Ngành ICT thiếu lịch sử nên loại khỏi thí nghiệm.

In [ ]:
from fetch_supplemental import run_from_cache, RAW_DIR, DEFAULT_OUTPUT
audit=run_from_cache(RAW_DIR,DEFAULT_OUTPUT)
print(json.dumps(audit,ensure_ascii=False,indent=2))
supp=pd.read_csv(DEFAULT_OUTPUT)
display(supp.tail())

## Kiểm tra độ phủ

Ô thiếu đầu chuỗi là thông tin chưa có, không tự thay bằng số của tương lai.

In [ ]:
display(supp.isna().sum().to_frame('Số ô thiếu'))